# The existing PDB loader in mBuild does not carry bond orders or formal charges
### Joseph R. Laforet Jr.

`mb.load` reads a PDB file through mdtraj. `Protein` reads the same file
through the residue definitions. This notebook showcases why the existing loader is insufficient for exporting to force field parameterization software like OpenFF or RDKit.

In [ ]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [ ]:
from collections import Counter

import mbuild as mb
from mbuild.biopolymers import Protein

mbuild_compound = mb.load("../semaglutide_apo.pdb")
mbuild_protein = Protein("../semaglutide_apo.pdb")


def bond_orders(compound):
    return dict(Counter(d["bond_order"] for *_, d in compound.bonds(return_bond_order=True)))


print("mb.load :", mbuild_compound.n_particles, "atoms", mbuild_compound.n_bonds, "bonds", bond_orders(mbuild_compound))
print("Protein :", mbuild_protein.n_particles, "atoms", mbuild_protein.n_bonds, "bonds", bond_orders(mbuild_protein))

In [ ]:
print("mb.load particle charges:", dict(Counter(p.charge for p in mbuild_compound.particles())))
print("Protein net formal charge:", mbuild_protein.net_formal_charge)
print({f"{r.name}{r.resnum}": r.formal_charge for r in mbuild_protein.residues() if r.formal_charge})

Only one of the two can become an OpenFF `Molecule`.

In [ ]:
from openff.toolkit import Molecule

openff_molecule = Molecule.from_rdkit(mbuild_protein.to_rdkit(), allow_undefined_stereo=True)
print("from Protein:", openff_molecule.n_atoms, "atoms, net charge", openff_molecule.total_charge)

try:
    Molecule.from_rdkit(mbuild_compound.to_rdkit(), allow_undefined_stereo=True)
except Exception as error:
    print("from mb.load:", type(error).__name__, str(error).splitlines()[0])